# Optimizing the `specify` skill with SkillOpt — structural ⊕ trace-judge

**What this is.** A real-data walkthrough of the **second** SkillOpt slice in this repo: optimizing
a `specify` (spec-writing) skill where the reward is **two layers** — a deterministic *structural*
checker **⊕** an LLM **trace-judge**. The headline result (**SC-006**) is that the judge adds signal
the deterministic checker physically cannot produce: every spec passes structure, yet the judge still
catches *untestable* requirements.

Everything below is read from a **real run** — `jobs/specify-skillopt-2026-05-29/` (33 LLM calls,
40,966 tokens, 412.7 s wall). No fabricated numbers; sources are listed at the bottom.

## How to read this — the dual reward

The `reviewer` slice used a *purely deterministic* reward (planted-bug F1). The `specify` slice
can't: **structure** is checkable by code, but **whether each requirement is genuinely testable**
is semantic. So the reward blends the two:

```
reward = 0.5 · structural(spec)  ⊕  0.5 · judge(brief, spec)
         └─ regex checks:           └─ LLM trace-judge scores:
            user-scenarios,            testable / coverage / outcome
            FR ids, MUST verbs,        (the layer code can't replace)
            measurable SC, …
```

`structural.soft` saturates near 1.0 once the agent emits a well-formed spec — so on a good spec the
**judge is the only live gradient**. Demonstrating that gap is exactly what SC-006 asks for.

In [1]:
import sys
from pathlib import Path

# import-path setup only — all analysis helpers live in notebooks/utils, never inlined.
_p = Path.cwd().resolve()
_root = next((d for d in (_p, *_p.parents) if (d / "pyproject.toml").exists()), _p)
sys.path.insert(0, str(_root / "notebooks"))
sys.path.insert(0, str(_root / "tasks" / "specify-skillopt"))

from utils import repo_root
from utils.skillopt_analysis import load_json, load_jsonl, mean, sc006_evidence

ROOT = repo_root()
CASE = ROOT / "tasks" / "specify-skillopt"
_jobs = sorted((ROOT / "jobs").glob("specify-skillopt-*"))
JOB = _jobs[-1] if _jobs else ROOT / "jobs" / "specify-skillopt-2026-05-29"

print("repo:", ROOT)
print("case:", CASE.relative_to(ROOT))
print("job :", JOB.relative_to(ROOT), "—", "found" if JOB.exists() else "MISSING (run the smoke first)")

repo: /Users/mhuang/Documents/GitHub/agentbook
case: tasks/specify-skillopt
job : jobs/specify-skillopt-2026-05-29 — found


## Step 0 — the target: a weak `specify` seed + the *structural* layer

The seed skill is a deliberately thin spec-writer. The structural reward is pure `re` (no LLM): it
scores **form** — User Scenarios, ≥3 `FR-` ids, `MUST` verbs, measurable Success Criteria, no
implementation leakage. Crucially it also declares `needs_judge`: the semantic aspects it *cannot*
verify and explicitly hands off to the trace-judge.

In [2]:
from specify_env.reward import reward  # pure stdlib (re) — no skillopt / no LLM

print("───── seed_skill.md (the weak optimization target) ─────")
print((CASE / "seed_skill.md").read_text().strip()[:900], "…\n")

# Structural reward on a REAL produced spec (the seed agent's saved rollout for spec_0006):
inst = next(i for i in load_json(CASE / "data/specify_split/test/items.json") if i["id"] == "spec_0006")
spec_md = load_json(JOB / "test_eval_baseline/predictions/spec_0006/conversation.json")[0]["content"]
r = reward(inst["brief"], spec_md)

print(f"brief: {inst['brief']}\n")
print(f"structural reward on the produced spec:  hard={r['hard']}  soft={r['soft']}")
print("checks:", "  ".join(f"{'✓' if v else '✗'} {k}" for k, v in r["checks"].items()))
print("\nneeds_judge — what the structural checker CANNOT verify (routed to the LLM):")
for nj in r["needs_judge"]:
    print("  -", nj)

───── seed_skill.md (the weak optimization target) ─────
# Specification Skill (seed — deliberately weak)

## Overview
Given a short feature brief, write a clear specification in Markdown describing what
should be built.

## Output
Write a spec covering the feature. Describe the intended behavior and what success
looks like. Be concrete and avoid leaving things vague.

<!--
HEADROOM (intentionally omitted from this seed; SkillOpt should re-derive it from
failing structural checks + the LLM trace-judge):
  - Required sections: "## User Scenarios & Testing" (with per-story "(Priority: Pn)"),
    "## Requirements" → "### Functional Requirements", "## Success Criteria"
    → "### Measurable Outcomes", "## Assumptions".
  - Every functional requirement gets a stable id ("**FR-001**") and is phrased with
    MUST, stating a testable, observable behavior — not an implementation.
  - Success criteria are measurable (numbers / thresholds / %), outcome- …

brief: A permissions layer where worksp

## Step 1 — held-out TEST: structure saturates, the judge discriminates

The run evaluated the seed (baseline) on the 3 held-out test specs. Below, per instance:
`structural_soft` (code) vs the judge's `testable / coverage / outcome`. Watch structure pin to
**1.00** while the judge stays below it — structure has no gradient left to give.

In [3]:
base = load_jsonl(JOB / "test_eval_baseline/results.jsonl")

hdr = f"{'id':9} {'struct':>6} {'judge':>6} {'testable':>9} {'coverage':>9} {'outcome':>8}"
print(hdr)
print("-" * len(hdr))
for row in base:
    print(
        f"{row['id']:9} {row['structural_soft']:>6.2f} {row['judge_overall']:>6.2f} "
        f"{row['judge_testable']:>9.2f} {row['judge_coverage']:>9.2f} {row['judge_outcome']:>8.2f}"
    )
print("-" * len(hdr))
print(
    f"{'mean':9} {mean(base, 'structural_soft'):>6.2f} {mean(base, 'judge_overall'):>6.2f} "
    f"{mean(base, 'judge_testable'):>9.2f} {mean(base, 'judge_coverage'):>9.2f} {mean(base, 'judge_outcome'):>8.2f}"
)
print("\nstructural_soft is identically 1.00 — saturated, no gradient. The judge carries all signal.")

id        struct  judge  testable  coverage  outcome
----------------------------------------------------
spec_0006   1.00   0.91      0.91      0.93     0.88
spec_0002   1.00   0.88      0.88      0.92     0.85
spec_0005   1.00   0.88      0.88      0.91     0.85
----------------------------------------------------
mean        1.00   0.89      0.89      0.92     0.86

structural_soft is identically 1.00 — saturated, no gradient. The judge carries all signal.


## Step 2 — SC-006: a spec that structure passed but the judge caught

SC-006 needs **at least one rollout** where structural passed (1.00) yet the judge down-scored
`testable` / `coverage` / `outcome`. `sc006_evidence()` finds them; here is the strongest witness
(lowest `judge_overall`) with the judge's **actual reason**, verbatim.

In [4]:
witnesses = sc006_evidence(base)
print(f"SC-006 witnesses (structural==1.0 AND a judge dim < 1.0): {len(witnesses)}/{len(base)}\n")

w = witnesses[0]  # strongest = lowest judge_overall
print(f"witness: {w['id']}  —  brief: {w['task_description']}…\n")
print(f"  structural_soft = {w['structural_soft']:.2f}   (code says: well-formed spec, nothing wrong)")
print(f"  judge_overall   = {w['judge_overall']:.2f}")
print(f"    · testable    = {w['judge_testable']:.2f}")
print(f"    · coverage    = {w['judge_coverage']:.2f}")
print(f"    · outcome     = {w['judge_outcome']:.2f}   ← the gap structure cannot see\n")
print("  judge_notes (the semantic gap, verbatim):")
print("   ", w["judge_notes"])

SC-006 witnesses (structural==1.0 AND a judge dim < 1.0): 3/3

witness: spec_0002  —  brief: Write a feature specification for: A feature that lets users export their saved articles as a sing
le EPUB file, pre…

  structural_soft = 1.00   (code says: well-formed spec, nothing wrong)
  judge_overall   = 0.88
    · testable    = 0.88
    · coverage    = 0.92
    · outcome     = 0.85   ← the gap structure cannot see

  judge_notes (the semantic gap, verbatim):
    SC-05 relies on user-reported bugs rather than an automated, directly measurable signal, weakening outcome verifiability.


## Step 3 — SELECT + GATE: the honest null

SkillOpt proposed an edit (2 success-patches merged), re-evaluated on held-out test, and the gate
**rejected** it — the edit did not beat the seed. The result is a documented null, not a fabricated
"it improved." This is the anti-reward-hacking property working as intended.

In [5]:
summary = load_json(JOB / "summary.json")
b_soft, a_soft = summary["baseline_test_soft"], summary["test_soft"]

print(
    f"epochs={summary['config']['num_epochs']}  train={summary['config']['train_size']}  "
    f"test={len(base)}  wall={summary['total_wall_time_s']}s\n"
)
print(f"{'':16} {'seed (before)':>14} {'best (after)':>14}")
print(f"{'TEST structural':16} {summary['baseline_test_hard']:>14.3f} {summary['test_hard']:>14.3f}")
print(f"{'TEST soft (⊕)':16} {b_soft:>14.3f} {a_soft:>14.3f}")
print(
    f"\naccepts={summary['total_accepts']}  rejects={summary['total_rejects']}  "
    f"best_step={summary['best_step']} ({summary['best_origin']})"
)
print(f"gate decision: edit REJECTED — test soft {b_soft:.3f} → {a_soft:.3f} (not an improvement)")

epochs=1  train=5  test=3  wall=412.7s

                  seed (before)   best (after)
TEST structural           1.000          1.000
TEST soft (⊕)             0.945          0.932

accepts=0  rejects=1  best_step=0 (initial_skill)
gate decision: edit REJECTED — test soft 0.945 → 0.932 (not an improvement)


## Analysis — why this is the right outcome

**The dual reward works end-to-end** (rollout → structural ⊕ judge → reflect → patch → gate →
held-out test), Docker-free, on the `claude` backend, in ~7 minutes.

**SC-006 is demonstrated** — every held-out spec is a witness (see Step 2). On each one the
deterministic checker returns `structural_soft = 1.00` (saturated and blind), while the trace-judge
returns `judge_overall ≈ 0.88–0.91` and names a concrete testability gap in its `judge_notes` — e.g.
a success criterion that "relies on user-reported bugs rather than an automated, directly measurable
signal." That is semantic signal no regex can produce, which is the whole reason the judge layer
earns its tokens. (The exact strongest-witness id and note are printed live in Step 2, so this
narrative tracks whatever run the bootstrap cell selected.)

**The gate rejected the edit (honest null).** Because structure was already maxed, the only headroom
was the judge's semantic axis — a single epoch on 5 train specs didn't find an edit that moved it on
held-out test (`soft 0.945 → 0.932`), so nothing was accepted. A fabricated "+X% improvement" would
have been the *wrong* result to report.

## Re-run it yourself (regenerates a fresh `jobs/` artifact)

`jobs/` is gitignored. From the repo root:

```bash
# 1. deps (once): SkillOpt editable into agentbook's env — no SkillOpt repo edits
conda run -p ./env pip install -e ../SkillOpt-src

# 2. real smoke run (~$1.5, ~10 min) → jobs/specify-skillopt-<date>/
conda run -p ./env python tasks/specify-skillopt/run_smoke.py
```

Then re-run this notebook top-to-bottom; the bootstrap cell auto-selects the newest
`jobs/specify-skillopt-*` directory.

## Data sources

Every number above is read from a **real run** — no fabricated data.

| What | Source (real) |
|---|---|
| Run artifacts (summary, history, per-instance test evals, predictions) | `jobs/specify-skillopt-2026-05-29/` (gitignored; regenerate via the cell above) |
| Optimization target skill (weak seed) | `tasks/specify-skillopt/seed_skill.md` |
| Deterministic structural reward (pure `re`) | `tasks/specify-skillopt/specify_env/reward.py` |
| LLM trace-judge (testable / coverage / outcome) | `tasks/specify-skillopt/specify_env/judge.py` |
| Held-out test briefs | `tasks/specify-skillopt/data/specify_split/test/items.json` |
| Analysis helpers (loaders, `sc006_evidence`) | `notebooks/utils/skillopt_analysis.py` |
| Spec / success criterion | `specs/skillopt-skill-optimization/spec.md` (SC-006), `tasks.md` (T025/T026) |